In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/notebooks
/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/app


In [4]:
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import queue

/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
model_size = "small" # Puedes usar "tiny" para más velocidad o "medium" para más precisión
model = WhisperModel(model_size, device="cuda")

In [10]:
sample_rate = 16000
audio_queue = queue.Queue()

In [11]:
def audio_callback(indata, frames, time, status):
    """Esta función captura el audio del micrófono y lo mete en una cola"""
    if status:
        print(status)
    audio_queue.put(indata.copy())

In [ ]:
print("--- Escuchando... (Presiona Ctrl+C para detener) ---")
with sd.InputStream(samplerate=sample_rate, channels=1, callback=audio_callback):
    while True:
        # Recuperamos el audio de la cola
        audio_chunk = audio_queue.get()
        
        # Convertimos a float32 (formato que entiende Whisper)
        audio_data = audio_chunk.flatten().astype(np.float32)
        
        # Transcribir el fragmento
        segments, info = model.transcribe(audio_data, beam_size=5, language="es")

        for segment in segments:
            print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

--- Escuchando... (Presiona Ctrl+C para detener) ---
